# DistilBERT Sequence Classification

In this notebook, we will transition from classical NLP to modern Deep Learning using Hugging Face Transformers.
We will be using PyTorch as our backend, which allows us to fully utilize your RTX 3050 GPU on Windows!

The workflow involves:
1. **Minimal Preprocessing**: Loading raw data and combining subject + body.
2. **AutoTokenizer**: Tokenizing text into `input_ids` and `attention_mask`.
3. **AutoModelForSequenceClassification**: Loading the pre-trained `distilbert-base-uncased` model.
4. **Trainer**: Fine-tuning the model on our specific customer ticket queues.


In [20]:
import pandas as pd
import numpy as np
import os
import torch
import evaluate

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

# Verify PyTorch can see the GPU!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


Using device: cuda
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Minimal Data Preprocessing
Transformers prefer raw text. We won't use stopwords removal or lemmatization, because models like BERT were trained on natural sentence structures. We'll simply load the data, handle missing values, and combine the subject and body.


In [21]:
# Load Data
data_path = "../artifacts/data_ingestion/dataset.csv"
df = pd.read_csv(data_path)

# Drop missing values in crucial columns
df = df.dropna(subset=['subject', 'body', 'queue'])

# Concatenate subject and body
df['text'] = df['subject'] + " " + df['body']

# Encode the target labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['queue'])
num_classes = len(le.classes_)

# Split into Train and Test
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].astype(str).tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    random_state=42
)

print(f"Training instances: {len(X_train_text)}")
print(f"Testing instances: {len(X_test_text)}")
print(f"Classes: {le.classes_}")


Training instances: 19799
Testing instances: 4950
Classes: ['Billing and Payments' 'Customer Service' 'General Inquiry'
 'Human Resources' 'IT Support' 'Product Support' 'Returns and Exchanges'
 'Sales and Pre-Sales' 'Service Outages and Maintenance'
 'Technical Support']


## 2. AutoTokenizer
Here we load the tokenizer. Notice how the tokenizer automatically converts words into `input_ids` and adds special tokens like `[CLS]` (101) and `[SEP]` (102).


In [22]:
# Load the DistilBERT Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Let's see it in action!
sample_text = "Payment failed today."
print(f"Original Text: {sample_text}\n")

tokens = tokenizer(sample_text)
print(f"input_ids: {tokens['input_ids']}")
print(f"attention_mask: {tokens['attention_mask']}\n")

# Let's decode it back to see the special tokens!
print(f"Decoded: {tokenizer.decode(tokens['input_ids'])}")


Original Text: Payment failed today.

input_ids: [101, 7909, 3478, 2651, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1]

Decoded: [CLS] payment failed today. [SEP]


## 3. Formatting Data for Hugging Face
The `Trainer` API expects data in a specific format called a `Dataset`. We will write a small tokenization function and apply it to our train and test splits.


In [23]:
# Create Hugging Face Datasets
train_dataset = Dataset.from_dict({'text': X_train_text, 'label': y_train})
test_dataset = Dataset.from_dict({'text': X_test_text, 'label': y_test})

# Tokenization function
def tokenize_function(examples):
    # Padding and truncation happens here. Max length for BERT is 512, 
    # but tickets are usually short. 128 is a good, fast balance.
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization to the datasets
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# We can remove the raw 'text' column now, the model only wants input_ids and attention_mask
train_tokenized = train_tokenized.remove_columns(["text"])
test_tokenized = test_tokenized.remove_columns(["text"])

train_tokenized.set_format("torch")
test_tokenized.set_format("torch")

print("Data formatted successfully!")


Map:   0%|          | 0/19799 [00:00<?, ? examples/s]

Map:   0%|          | 0/4950 [00:00<?, ? examples/s]

Data formatted successfully!


## 4. AutoModelForSequenceClassification
We now load the pre-trained DistilBERT model and attach a linear classification head configured for our specific number of queues.


In [24]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=num_classes
)

# Move model to GPU
model.to(device)

print(f"Model loaded and moved to {device}!")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to cuda!


## 5. Trainer and TrainingArguments
The `Trainer` automates the entire fine-tuning loop. We define `TrainingArguments` to set batch size, epochs, learning rate, and logging configurations. 

We will also define a metric calculation function using the `evaluate` library to track our Macro F1 score and Accuracy during training!


In [25]:
## 5. Custom Trainer for Imbalanced Data
import torch
from torch import nn
from transformers import Trainer, TrainingArguments
import evaluate
from sklearn.utils.class_weight import compute_class_weight

# 1. Compute class weights just like we did for the classical models!
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# 2. Subclass the Trainer to inject our class weights into the Loss Function
class ImbalancedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Apply CrossEntropyLoss with our balancing weights!
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 3. Define Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "macro_f1": f1["f1"]}

# 4. Training Arguments
os.makedirs("../artifacts/models/distilbert", exist_ok=True)

training_args = TrainingArguments(
    output_dir="../artifacts/models/distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # Slightly bumped learning rate
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,          # Bumped to 5 epochs for better convergence
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=50,
)

# 5. Initialize our Custom Trainer
trainer = ImbalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

print("Starting Fine-Tuning with Class Weights!")
trainer.train()



[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting Fine-Tuning with Class Weights!


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.857010,1.860462,0.302828,0.271906
2,1.679766,1.735511,0.292929,0.283687
3,1.488933,1.631142,0.367879,0.361843
4,1.145516,1.608906,0.398182,0.394935
5,0.965655,1.618327,0.395960,0.396216


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6190, training_loss=1.4872225457901715, metrics={'train_runtime': 724.6829, 'train_samples_per_second': 136.605, 'train_steps_per_second': 8.542, 'total_flos': 3278870257728000.0, 'train_loss': 1.4872225457901715, 'epoch': 5.0})

In [26]:
# 6. Evaluation and MLflow Logging
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("Evaluating model on Test Set...")
# The Trainer makes predictions seamlessly on the tokenized dataset!
predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=1)

# Configure MLflow
db_path = os.path.abspath("../mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("Customer Support Ticket Classification")

print("Logging results to MLflow...")
with mlflow.start_run(run_name="DistilBERT_FineTuned"):
    
    # Calculate classical metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    # Log metrics to MLflow
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("macro_precision", prec)
    mlflow.log_metric("macro_recall", rec)
    mlflow.log_metric("macro_f1", f1)
    
    # Print and Log Text Report
    report = classification_report(y_test, y_pred, target_names=le.classes_)
    print("\nClassification Report:\n", report)
    
    with open("distilbert_classification_report.txt", "w") as f:
        f.write(report)
    mlflow.log_artifact("distilbert_classification_report.txt", artifact_path="evaluation_metrics")
    
    # Generate and Log Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10,7))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('DistilBERT Confusion Matrix')
    plt.savefig("distilbert_confusion_matrix.png")
    plt.close()
    mlflow.log_artifact("distilbert_confusion_matrix.png", artifact_path="evaluation_metrics")
    
    print("\nSuccessfully logged to MLflow! Go check the UI to compare against BiLSTM and LinearSVC!")


ModuleNotFoundError: No module named 'mlflow'